# Hather: teacher Run B (label every message)

The teacher reads each formatted message (`[SOURCE]/[SENDER]/[SIGNALS]/[TEXT]`) and answers with
`verdict`, `confidence`, `signals` and a short `explanation`. It never sees the label. The answers become the student's training data.

**Jobs files** (built on your laptop with `data/make_runB_jobs.py`, uploaded as one private Kaggle dataset):

| Jobs file | What | When |
| --- | --- | --- |
| `runB_compare_jobs.jsonl` | 100 English test SMS | Done by `teacher_prep.ipynb` (both teachers in one run); `prep_summary.md` names the winner |
| `runB_jobs_v2.jsonl` | the v2 training pool (about 9,750 messages with the rewrites) | Then, with the winning teacher |
| `runB_test_jobs.jsonl` | the frozen test set | Later, to measure the teacher in Phase 4 (never trained on) |

**Before running:** Accelerator **GPU T4 x2**, Internet **on**, add the jobs dataset and `llamacpp-t4` as inputs.
To continue a stopped run, also add that version's output; finished jobs are skipped.
Then *Save Version → Save & Run All* and check the Logs for "server ready" and "thinking is off, OK".

In [ ]:
# 0. Dead switch + settings: stops in seconds if anything is wrong, before the slow steps
NOTEBOOK_VERSION = "2026-10-25c (mixed-language rule)"
print("notebook version:", NOTEBOOK_VERSION)
import glob, os, json, time, re, subprocess, shutil, sys

JOBS_NAME = "runB_jobs_v2.jsonl"          # runB_jobs_v2.jsonl (training pool) -> later runB_test_jobs.jsonl
TEACHER   = "original"                    # winner of the Oct 24 comparison (98% vs 96%): "abliterated" or "original"

TEACHERS = {
    "abliterated": ("huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF", "Huihui-Qwen3.8-27B-abliterated-UD-DW-Q4_K_M.gguf"),
    "original":    ("unsloth/Qwen3.8-27B-GGUF", "Qwen3.8-27B-UD-Q4_K_M.gguf"),
}
MODEL_REPO, MODEL_FILE = TEACHERS[TEACHER]
PARALLEL     = 8          # Run B answers are short, so more requests at once pay off
CTX_PER_SLOT = 2048       # system prompt + message + answer fit in about 1,000 tokens
PORT = 8080
STEM  = re.sub(r"_jobs(_v\d+)?\.jsonl$", r"\1", JOBS_NAME) + f"_{TEACHER}"   # e.g. runB_v2_abliterated
OUT   = f"/kaggle/working/{STEM}_out.jsonl"                       # one line per finished job
FINAL = f"/kaggle/working/{STEM}_labels.jsonl"                    # cleaned answers + meta

problems = []
found = glob.glob(f"/kaggle/input/**/{JOBS_NAME}", recursive=True)
if not found:
    problems.append(f"{JOBS_NAME} not found under /kaggle/input (Add Input -> your jobs dataset)")
else:
    JOBS = found[0]
    try:
        ALL_JOBS = [json.loads(l) for l in open(JOBS, encoding="utf-8") if l.strip()]
        bad = [i for i, j in enumerate(ALL_JOBS) if not {"job_id", "messages", "schema", "sampling", "meta"} <= j.keys()]
        if bad: problems.append(f"{len(bad)} malformed jobs, first at line {bad[0]+1}")
        if len({j["job_id"] for j in ALL_JOBS}) != len(ALL_JOBS): problems.append("duplicate job_ids")
    except Exception as e:
        problems.append(f"cannot read {JOBS}: {e}")
try:
    open("/kaggle/working/.write_test", "w").write("ok"); os.remove("/kaggle/working/.write_test")
except Exception as e:
    problems.append(f"/kaggle/working not writable: {e}")
gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
if len(gpus) < 2: problems.append(f"need GPU T4 x2, found: {gpus or 'no GPU'}")
if shutil.disk_usage("/tmp").free < 20e9: problems.append("less than 20 GB free in /tmp for the model")

if problems:
    print("STOPPING:\n - " + "\n - ".join(problems)); raise SystemExit(1)
print(f"OK: {len(ALL_JOBS)} jobs from {JOBS}\nteacher: {TEACHER} ({MODEL_FILE})\noutput: {OUT}\nGPUs: {gpus}")

In [ ]:
# 1. llama.cpp: reuse the llamacpp-t4 dataset if attached, otherwise build the newest release (~26 min)
found = glob.glob("/kaggle/input/**/llama-server", recursive=True)
BIN = "/kaggle/working/llamacpp-bin"
if found:
    shutil.copytree(os.path.dirname(found[0]), BIN, dirs_exist_ok=True)
    print("using prebuilt llama.cpp from", found[0])
else:
    import urllib.request
    tag = json.load(urllib.request.urlopen("https://api.github.com/repos/ggml-org/llama.cpp/releases/latest"))["tag_name"]
    print("building llama.cpp", tag)
    !git clone --depth 1 --branch {tag} https://github.com/ggml-org/llama.cpp /tmp/llama.cpp > /kaggle/working/build.log 2>&1
    # GGML_CUDA_NO_VMM=ON is required: Kaggle's image has no libcuda.so for CMake to link
    !cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DGGML_CUDA_NO_VMM=ON -DBUILD_SHARED_LIBS=OFF -DLLAMA_CURL=OFF >> /kaggle/working/build.log 2>&1
    !cmake --build /tmp/llama.cpp/build --config Release -j 4 --target llama-server >> /kaggle/working/build.log 2>&1
    os.makedirs(BIN, exist_ok=True)
    shutil.copy("/tmp/llama.cpp/build/bin/llama-server", BIN)
    print("built. Save this notebook's output as the llamacpp-t4 dataset to skip the build next time.")
os.chmod(f"{BIN}/llama-server", 0o755)

In [ ]:
# 2. Download the teacher into /tmp (Kaggle's saved folder is only ~20 GB)
from huggingface_hub import hf_hub_download
t = time.time()
MODEL = hf_hub_download(MODEL_REPO, MODEL_FILE, local_dir="/tmp/models")
print(MODEL, f"{os.path.getsize(MODEL)/1e9:.2f} GB in {time.time()-t:.0f}s")

In [ ]:
# 3. Start the server with thinking OFF, then one smoke-test request
cmd = [f"{BIN}/llama-server", "-m", MODEL, "-ngl", "99", "--split-mode", "layer",
       "-c", str(CTX_PER_SLOT * PARALLEL), "--parallel", str(PARALLEL),
       "--jinja", "--chat-template-kwargs", '{"enable_thinking":false}',
       "--port", str(PORT)]
server = subprocess.Popen(cmd, stdout=open("/kaggle/working/server.log", "w"), stderr=subprocess.STDOUT)

import urllib.request
for i in range(180):                                   # up to 15 min to load
    try:
        if urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=5).status == 200:
            print(f"server ready after {i*5}s"); break
    except Exception:
        pass
    if server.poll() is not None:
        print(open("/kaggle/working/server.log").read()[-3000:]); raise SystemExit("server crashed")
    time.sleep(5)
else:
    raise SystemExit("server did not start; see server.log")

job = ALL_JOBS[0]
body = {"model": "teacher", "messages": job["messages"], **job["sampling"],
        "response_format": {"type": "json_schema", "json_schema": {"name": "out", "schema": job["schema"]}},
        "chat_template_kwargs": {"enable_thinking": False}}
req = urllib.request.Request(f"http://localhost:{PORT}/v1/chat/completions", data=json.dumps(body).encode(),
                             headers={"Content-Type": "application/json"})
try:
    msg = json.load(urllib.request.urlopen(req, timeout=600))["choices"][0]["message"]
except urllib.error.HTTPError as e:
    raise SystemExit(f"smoke test rejected: HTTP {e.code} {e.read()[:500]}")
if msg.get("reasoning_content") or "<think>" in (msg.get("content") or ""):
    raise SystemExit("the model is still thinking: check --chat-template-kwargs above")
print("thinking is off, OK\n", msg["content"])
json.loads(msg["content"])

In [ ]:
# 4. The client: PARALLEL requests at a time, JSON schema enforced, every answer saved immediately
import asyncio, httpx
from collections import Counter

URL = f"http://localhost:{PORT}/v1/chat/completions"
ARABIC = re.compile(r"[؀-ۿ]")

def arabic_ratio(text):
    letters = [c for c in text if c.isalpha()]
    return sum(bool(ARABIC.match(c)) for c in letters) / max(1, len(letters))

def message_text(job):
    return job["messages"][-1]["content"].split("[TEXT]", 1)[-1].strip()

def check(job, data):
    """Validate one answer. Returns (answer, problems); problems are recorded, not fatal."""
    if data.get("verdict") not in ("scam", "legit"): raise ValueError(f"bad verdict {data.get('verdict')!r}")
    conf = data.get("confidence")
    if not isinstance(conf, int) or not 0 <= conf <= 100: raise ValueError(f"bad confidence {conf!r}")
    expl = (data.get("explanation") or "").strip()
    if not expl: raise ValueError("empty explanation")
    answer = {"verdict": data["verdict"], "confidence": conf,
              "signals": [s.strip() for s in data.get("signals", []) if s.strip()][:5], "explanation": expl}
    problems = []
    if len(expl.split()) > 40: problems.append("long_explanation")
    # Language rule: a clearly Arabic message (80%+ Arabic letters) needs an Arabic explanation, a clearly
    # English one (under 20%) an English one. Mixed messages (many Saudi company SMS are bilingual) may be
    # explained in either language.
    r_msg, expl_ar = arabic_ratio(message_text(job)), arabic_ratio(expl) >= 0.5
    if (r_msg >= 0.8 and not expl_ar) or (r_msg < 0.2 and expl_ar): problems.append("wrong_language")
    return answer, problems

def load_done(*paths):
    done = set()
    for p in paths:
        for line in open(p, encoding="utf-8"):
            r = json.loads(line)
            if r.get("ok"): done.add(r["job_id"])
    return done

async def run(jobs, out_path=OUT, parallel=PARALLEL, url=URL, retries=3):
    name = os.path.basename(out_path)
    previous = [p for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True)]
    if previous and not os.path.exists(out_path):          # carry over answers from an attached earlier run
        with open(out_path, "w", encoding="utf-8") as f:
            for p in previous:
                f.writelines(l for l in open(p, encoding="utf-8") if json.loads(l).get("ok"))
    done = load_done(out_path) if os.path.exists(out_path) else set()
    todo = [j for j in jobs if j["job_id"] not in done]
    print(f"{len(done)} already done, {len(todo)} to go")
    if not todo: return
    sem, lock = asyncio.Semaphore(parallel), asyncio.Lock()
    stats = {"n": 0, "fail": 0, "t0": time.time()}
    out = open(out_path, "a", encoding="utf-8")

    async def one(client, job):
        async with sem:
            rec = {"job_id": job["job_id"], "ok": False}
            for attempt in range(retries):
                t = time.time()
                try:
                    body = {"model": "teacher", "messages": job["messages"], **job["sampling"],
                            "response_format": {"type": "json_schema",
                                                "json_schema": {"name": "out", "schema": job["schema"]}},
                            "chat_template_kwargs": {"enable_thinking": False}}
                    r = await client.post(url, json=body, timeout=600)
                    if r.status_code != 200:
                        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:300]}")
                    msg = r.json()["choices"][0]["message"]
                    content = re.sub(r"<think>.*?</think>", "", msg.get("content") or "", flags=re.S).strip()
                    answer, problems = check(job, json.loads(content))
                    rec = {"job_id": job["job_id"], "ok": True, "answer": answer, "problems": problems,
                           "seconds": round(time.time() - t, 1),
                           "thinking": bool(msg.get("reasoning_content"))}
                    break
                except Exception as e:
                    rec["error"] = f"{type(e).__name__}: {str(e)[:300]}"
            async with lock:
                out.write(json.dumps(rec, ensure_ascii=False) + "\n"); out.flush()
                stats["n"] += 1; stats["fail"] += not rec["ok"]
                if stats["n"] % 100 == 0 or stats["n"] == len(todo):
                    el = time.time() - stats["t0"]; rate = stats["n"] / el
                    print(f"{stats['n']}/{len(todo)}  failed={stats['fail']}  {rate*60:.1f} jobs/min  "
                          f"ETA {(len(todo)-stats['n'])/rate/60:.0f} min", flush=True)

    async with httpx.AsyncClient(limits=httpx.Limits(max_connections=parallel)) as client:
        await asyncio.gather(*(one(client, j) for j in todo))
    out.close()

def results(path=OUT):
    """Latest answer per job_id, joined to the job's meta."""
    jobs = {j["job_id"]: j for j in ALL_JOBS}
    latest = {}
    for line in open(path, encoding="utf-8"):
        r = json.loads(line)
        if r["job_id"] in jobs and (r.get("ok") or r["job_id"] not in latest):
            latest[r["job_id"]] = r
    return [{**r, "meta": jobs[k]["meta"]} for k, r in latest.items()]

def report(rows):
    ok = [r for r in rows if r.get("ok")]
    print(f"{len(ok)} answered, {len(rows)-len(ok)} failed")
    groups = Counter(); agree = Counter()
    for r in ok:
        m = r["meta"]; key = (m.get("origin", "?"), m.get("language", "?"), m.get("label") or "?")
        groups[key] += 1; agree[key] += r["answer"]["verdict"] == m.get("label")
    print(f"\n{'origin/language/label':40} {'n':>6} {'agree':>7}")
    for k in sorted(groups):
        print(f"{'/'.join(k):40} {groups[k]:>6} {agree[k]/groups[k]:>7.1%}")
    tot = sum(groups.values())
    if tot:
        print(f"{'ALL':40} {tot:>6} {sum(agree.values())/tot:>7.1%}")
    print("problems:", dict(Counter(p for r in ok for p in r["problems"])))
    if any(r.get("thinking") for r in ok): print("WARNING: some answers contain thinking text")
    errs = Counter(r.get("error", "")[:80] for r in rows if not r.get("ok"))
    if errs: print("errors:", errs.most_common(5))

print(len(ALL_JOBS), "jobs loaded")

In [ ]:
# 5. Timing test on a spread sample (10 jobs from each origin/language/label group), then an estimate.
#    Stops the run here if the answers come back in the wrong language too often (seen in the Oct 24
#    comparison: English scams explained in Arabic), so a night of GPU time is not wasted.
import random
groups = {}
for j in ALL_JOBS:
    m = j["meta"]; groups.setdefault((m.get("origin"), m.get("language"), m.get("label")), []).append(j)
SAMPLE = [j for g in sorted(groups, key=str) for j in random.Random(0).sample(groups[g], min(10, len(groups[g])))]
t = time.time()
await run(SAMPLE)
per_job = (time.time() - t) / len(SAMPLE)
print(f"~{per_job:.1f}s per job with {PARALLEL} in parallel -> whole run ~{per_job*len(ALL_JOBS)/3600:.1f} hours")
rows = results()
jobs_by_id = {j["job_id"]: j for j in ALL_JOBS}
for r in rows:                      # re-check with the current rules (answers carried over from an older run)
    if r.get("ok"):
        r["answer"], r["problems"] = check(jobs_by_id[r["job_id"]], r["answer"])
report(rows)
ok = [r for r in rows if r.get("ok")]
if not ok:
    raise SystemExit("no test job succeeded: see the errors above and server.log")
for lang, clear in (("en", lambda x: x < 0.2), ("ar", lambda x: x >= 0.8)):   # mixed messages may use either
    lr = [r for r in ok if clear(arabic_ratio(message_text(jobs_by_id[r["job_id"]])))]
    wl = sum("wrong_language" in r["problems"] for r in lr)
    print(f"wrong_language on clearly {lang} messages: {wl}/{len(lr)}")
    if lr and wl / len(lr) > 0.15:
        raise SystemExit(f"STOPPING: {wl} of {len(lr)} clearly {lang} messages got an answer in the wrong language. "
                         "Fix the system prompt in data/make_runB_jobs.py before the full run.")
if per_job * len(ALL_JOBS) > 11.5 * 3600:
    print("NOTE: the full run will not fit in one 12-hour session. Let it run; then attach this version's "
          "output and run again to finish the rest.")
for r in ok[:4]:
    print(f"\n[{r['meta'].get('label')}] {json.dumps(r['answer'], ensure_ascii=False)}")

In [ ]:
# 6. The full run (skips everything already done). If it hits the 12-hour limit, attach this
#    version's output and run again: it continues where it stopped.
await run(ALL_JOBS)

In [ ]:
# 7. Final report and the labels file (download it into hather/data/out/)
rows = results()
jobs_by_id = {j["job_id"]: j for j in ALL_JOBS}
for r in rows:                      # re-check with the current rules (answers carried over from an older run)
    if r.get("ok"):
        r["answer"], r["problems"] = check(jobs_by_id[r["job_id"]], r["answer"])
report(rows)
with open(FINAL, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps({"id": r["job_id"], "ok": r.get("ok", False), "teacher": TEACHER,
                            "answer": r.get("answer"), "problems": r.get("problems", []),
                            "error": r.get("error"), "meta": r["meta"]}, ensure_ascii=False) + "\n")
print(f"\n{len(rows)} rows -> {FINAL}")

import random
wrong = [r for r in rows if r.get("ok") and r["answer"]["verdict"] != r["meta"].get("label")]
print(f"\n{len(wrong)} disagreements, a few of them:")
for r in random.sample(wrong, min(6, len(wrong))):
    job = next(j for j in ALL_JOBS if j["job_id"] == r["job_id"])
    print(f"\n--- label {r['meta'].get('label')} | teacher {r['answer']['verdict']} {r['answer']['confidence']}\n"
          f"{job['messages'][-1]['content'][:400]}\n> {r['answer']['explanation']}")

**When it finishes:** download `<jobs>_<teacher>_labels.jsonl` from the Output tab into `hather/data/out/`.
If any jobs failed, run again with this version's output attached; only the failures are redone.

- After the **compare** runs (one per teacher), pick the teacher with the higher `ALL` agreement.
- After the **full** run: `python data/split_runB.py data/out/runB_v2_<teacher>_labels.jsonl data/out/runB_pool_v2.jsonl`
  keeps the answers that match the labels and writes `train.jsonl` and `val.jsonl`.